In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Zpětná propagace operátoru (OBP) pro odhad střední hodnoty

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Odhadovaná doba použití: 4 minuty na procesoru Heron r3 (POZNÁMKA: Jedná se pouze o odhad. Skutečná doba běhu se může lišit.)*
## Výsledky učení
Po absolvování tohoto tutoriálu by uživatelé měli rozumět:
- Jak používat [`qiskit-addon-obp`](https://github.com/Qiskit/qiskit-addon-obp) ke snížení hloubky kvantového Circuit za cenu zvýšeného počtu spuštění Circuit
- Jak používat [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) k vytváření XYZ hamiltonianů a jejich Circuit časového vývoje

## Předpoklady
Doporučujeme, aby uživatelé před tímto tutoriálem ovládali tato témata:
- Použití primitivy [Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) pro výpočet středních hodnot pozorovatelné veličiny

## Pozadí
Zpětná propagace operátoru je technika, která spočívá v absorpci operací z konce kvantového Circuit do měřené pozorovatelné veličiny. Obecně tak snižuje hloubku Circuit za cenu většího počtu členů v pozorovatelné veličině. Cílem je zpětně propagovat co největší část Circuit bez toho, aby pozorovatelná veličina příliš narostla. Implementace v Qiskitu je k dispozici v doplňku OBP Qiskit. Podrobnosti najdeš v příslušné [dokumentaci](https://qiskit.github.io/qiskit-addon-obp/).

Uvažujme příkladový Circuit, pro nějž má být změřena pozorovatelná veličina $O = \sum_P c_P P$, kde $P$ jsou Pauliho operátory a $c_P$ jsou koeficienty. Circuit označíme jako jedinou unitární transformaci $U$, kterou lze logicky rozdělit na $U = U_C U_Q$, jak je znázorněno na obrázku níže.

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Zpětná propagace operátoru absorbuje unitární transformaci $U_C$ do pozorovatelné veličiny jejím vyvinutím jako $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Jinými slovy, část výpočtu se provede klasicky pomocí vývoje pozorovatelné veličiny z $O$ na $O'$. Původní problém lze nyní přeformulovat jako měření pozorovatelné veličiny $O'$ pro nový Circuit s menší hloubkou, jehož unitární transformace je $U_Q$.

Unitární transformace $U_C$ je reprezentována jako počet řezů $U_C = U_S U_{S-1}...U_2U_1$. Existuje více způsobů, jak definovat řez. Například ve výše uvedeném příkladovém Circuit lze každou vrstvu $R_{zz}$ a každou vrstvu Gate $R_x$ považovat za samostatný řez. Zpětná propagace zahrnuje klasický výpočet $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$. Každý řez $U_s$ lze vyjádřit jako $U_s = exp(\frac{-i\theta_s P_s}{2})$, kde $P_s$ je $n$-qubitový Pauliho operátor a $\theta_s$ je skalár. Snadno ověříme, že

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$

$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Ve výše uvedeném příkladu, pokud ${P,P_s} = 0$, potřebujeme ke výpočtu střední hodnoty spustit dva kvantové Circuit místo jednoho. Proto zpětná propagace může zvýšit počet členů v pozorovatelné veličině, což vede k vyššímu počtu spuštění Circuit. Jedním ze způsobů, jak umožnit hlubší zpětnou propagaci do Circuit a zároveň zabránit příliš velkému nárůstu operátoru, je ořezat členy s malými koeficienty namísto jejich přidání do operátoru. Například v uvedeném příkladu lze zvolit ořezání členu obsahujícího $P_sP$ za předpokladu, že $\theta_s$ je dostatečně malý. Ořezání členů může vést k menšímu počtu kvantových Circuit, ale způsobuje určitou chybu ve výsledném výpočtu střední hodnoty úměrnou velikosti koeficientů ořezaných členů.
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

- Qiskit SDK v2.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 nebo novější (`pip install qiskit-ibm-runtime`)
- Doplněk OBP Qiskit 0.3 nebo novější (`pip install qiskit-addon-obp`)
- Pomocné nástroje Qiskit addon 0.3 nebo novější (`pip install qiskit-addon-utils`)
## Nastavení

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Příklad na simulátoru v malém měřítku
Tento tutoriál implementuje [vzor Qiskit](/guides/intro-to-patterns) pro simulaci kvantové dynamiky Heisenbergova spinového řetězce pomocí [doplňku OBP Qiskit](https://github.com/Qiskit/qiskit-addon-obp). Všimni si, že na bezešumém simulátoru bude střední hodnota získaná s i bez zpětné propagace stejná.
### Krok 1: Mapování klasických vstupů na kvantový problém
#### Mapování časového vývoje kvantového Heisenbergova modelu na kvantový experiment
Nejprve použijeme funkci [`generate_xyz_hamiltonian`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian) z balíčku `qiskit-addon-utils` pro generování Heisenbergova hamiltoniánu na daném grafu konektivity. Tento graf může být buď [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) nebo [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap). V následující části použijeme lineární řetězec `CouplingMap` 10 Qubitů.

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Dále vygenerujeme Pauliho operátor modelující Heisenbergův hamiltonián XYZ:

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z}),}
$$

kde $G(V,E)$ je graf mapy propojení. Pro tento tutoriál jsme použili $J_x, J_y, J_z$ rovné $\frac{\pi}{8}, \frac{\pi}{4}, \frac{\pi}{2}$ a $h_x, h_y, h_z$ rovné $\frac{\pi}{3}, \frac{\pi}{6}, \frac{\pi}{9}$.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

From the qubit operator, we can generate a quantum circuit which models its time evolution. We have used [`generate_time_evolution_circuit`](/docs/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) with Lie Trotter decomposition to construct the time evolution circuit.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

Z qubitového operátoru můžeme vygenerovat kvantový Circuit modelující jeho časový vývoj. Použili jsme funkci [`generate_time_evolution_circuit`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) s Lie-Trotterovou dekompozicí pro sestavení Circuit časového vývoje.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif)

### Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
#### Vytvoření řezů Circuit pro zpětnou propagaci
Funkce `backpropagate` zpracovává vždy celé řezy Circuit najednou, takže způsob dělení může ovlivnit výkon zpětné propagace pro daný problém. Zde seskupíme Gate stejného typu do řezů pomocí funkce [`slice_by_depth`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_depth).

Podrobnější diskusi o dělení Circuit najdeš v tomto [návodu](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) balíčku [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils).

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

#### Backpropagate slices from the circuit

First we specify the observable to be $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, $N$ being the number of qubits. We will backpropagate slices from the time-evolution circuit until the terms in the observable can no longer be combined into eight or fewer qubit-wise commuting Pauli groups.

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

#### Omezení maximálního nárůstu operátoru během zpětné propagace
Během zpětné propagace počet členů v operátoru obecně rychle narůstá směrem k $2^L$, kde $L$ je počet řezů. Pokud dva členy operátoru vzájemně nekomutují po qubitech, potřebujeme k získání odpovídajících středních hodnot samostatné Circuit. Například pro dvouqubitovou pozorovatelnou veličinu $O = 0.1 XX + 0.3 IZ - 0.5 IX$ platí, že $[XX,IX] = 0$, takže k výpočtu středních hodnot těchto dvou členů stačí měření v jediné bázi. Avšak $IZ$ antikomutuje s oběma zbývajícími členy, takže potřebujeme samostatné měření v jiné bázi pro výpočet střední hodnoty $IZ$. Jinými slovy, k výpočtu $\langle O \rangle$ potřebujeme dva Circuit místo jednoho. S narůstajícím počtem členů v operátoru může narůstat i požadovaný počet spuštění Circuit.

Velikost operátoru lze omezit zadáním argumentu `operator_budget` funkce `backpropagate`, který přijímá instanci [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Abychom omezili množství přidělených zdrojů (počet spuštění Circuit, a tedy požadovaný čas QPU), omezíme maximální počet qubitově komutujících Pauliho skupin, které může mít zpětně propagovaná pozorovatelná veličina. Zde určíme, že zpětná propagace se zastaví, jakmile počet qubitově komutujících Pauliho skupin v operátoru překročí osm.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Zpětná propagace řezů z Circuit
Nejprve definujeme pozorovatelnou veličinu jako $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, kde $N$ je počet Qubitů. Budeme zpětně propagovat řezy z Circuit časového vývoje, dokud členy v pozorovatelné veličině nelze sloučit do osmi nebo méně qubitově komutujících Pauliho skupin.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

Níže uvidíš, že bylo zpětně propagováno šest řezů a členy byly sloučeny do šesti, nikoli osmi skupin. To znamená, že zpětná propagace dalšího řezu by způsobila překročení osmi Pauliho skupin. Tuto skutečnost můžeme ověřit prohlédnutím vrácených metadat. Všimni si také, že v této části je transformace Circuit přesná — žádné členy nové pozorovatelné veličiny $O'$ nebyly ořezány. Zpětně propagovaný Circuit a zpětně propagovaný operátor dávají přesně stejný výsledek jako původní Circuit a operátor.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif)

Pro příklad malého měřítku na simulátoru nebudeme používat ořezání. Důvodem je, že v nepřítomnosti šumu vede Circuit s i bez zpětné propagace ke stejnému výsledku a ořezání výsledek zhorší kvůli zavedené aproximaci.
#### Transpilace Circuit do sady bazových Gate
Nyní transpilujeme původní i zpětně propagovaný Circuit do bazových Gate Backend. Nepotřebujeme transpilovat na skutečném Backend, protože pro malý příklad poběžíme na simulátoru.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

As expected, the two expectation values agree. Because we are running on a noiseless statevector simulator, backpropagation is an exact transformation of the circuit-observable pair, so the original and backpropagated workflows must produce the same value of $M_Z$. The benefit of backpropagation only becomes apparent on noisy hardware, where the shorter backpropagated circuit accumulates less error, as illustrated in the large-scale hardware example below.

## Large-scale hardware example

When developing an experiment, it's useful to start with a small circuit to make visualizations and simulations easier. Now we look into operator backpropagation for a 50-qubit Heisenberg Hamiltonian with the same set of values for the $J$ and $h$ parameters and the same observable $M_Z$, but for four Trotter steps. The ideal expectation value at this scale cannot be calculated in a brute force method, so we use a tensor network and obtain the ideal expectation value to be $\simeq 0.89$.

Along with backpropagation, in this large-scale example, we also introduce backpropagation with truncation. Ideally we want to backpropagate as much as possible to reduce the depth of the effective circuit. However, it often leads to a large number of non-commuting terms in the updated observable, increasing the quantum overhead. Therefore, we can eliminate observable terms with small coefficients using a practice called truncation. While truncation allows more propagation by reducing the number of terms in the updated observable, it also introduces some approximation. Therefore, it is necessary to restrict the truncation within some limits so that the approximation error does not overwhelm the reduction in noise obtained from deeper backpropagation.

To restrict the amount of truncation, we allot an error budget for each slice as well as the total error budget over the entire backpropagated circuit using the [`setup_budget`](/docs/api/qiskit-addon-obp/utils-truncating#setup_budget) function. This ensures that the truncation is controlled for each slice as well as for the entire circuit. See also this [guide](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html) for other ways of allocating the budget.

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

### Krok 4: Následné zpracování a vrácení výsledku do požadovaného klasického formátu
Nyní získáme střední hodnoty původního a zpětně propagovaného Circuit.